In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
!pip install -U trl bitsandbytes

In [ ]:
!pip install -U transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 157.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transformers-5.12.0


In [ ]:
import json
import random
from pathlib import Path
from datasets import load_dataset
import random
import torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# Clean Dolly data

In [ ]:
DATA_DIR = Path("data/dolly_sft_full")
DATA_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_MESSAGE = "You are a helpful assistant."


In [ ]:
def build_user_message(instruction, context):
    instruction = (instruction or "").strip()
    context = (context or "").strip()

    if context:
        return f"Context:\n{context}\n\nInstruction:\n{instruction}"
    return instruction

def convert_example(example):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": build_user_message(example.get("instruction", ""), example.get("context", ""))},
            {"role": "assistant", "content": (example.get("response", "") or "").strip()},
        ],
        "category": example.get("category", "unknown")
    }

In [ ]:
def save_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [ ]:
raw = load_dataset("databricks/databricks-dolly-15k", split="train")
raw = raw.shuffle(seed=42)

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [ ]:
converted = [convert_example(ex) for ex in raw]

In [ ]:
train_data = converted[:13000]
valid_data = converted[13000:14000]
test_data  = converted[14000:]

In [ ]:
save_jsonl(train_data, DATA_DIR / "train.jsonl")
save_jsonl(valid_data, DATA_DIR / "valid.jsonl")
save_jsonl(test_data, DATA_DIR / "test.jsonl")

print(len(train_data), len(valid_data), len(test_data))

13000 1000 1011


In [ ]:
random.seed(42)

def make_eval_item(example):
    messages = example["messages"]
    return {
        "prompt_messages": messages[:-1],
        "target_answer": messages[-1]["content"],
        "category": example.get("category", "unknown")
    }

eval_50 = [make_eval_item(ex) for ex in random.sample(test_data, 50)]
save_jsonl(eval_50, DATA_DIR / "eval_50.jsonl")

# Evaluation Qwen2.5-1.5B-Instruct before Fine Tuning

In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
DATA_DIR =Path("data/dolly_sft_full")
EVAL_PATH = DATA_DIR / "eval_50.jsonl"

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "eval_50_output.jsonl"

In [ ]:
model.eval()

def generate_answer(prompt_messages):
    text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )
    return answer.strip()

In [ ]:
with open(EVAL_PATH, "r", encoding="utf-8") as f_in, open(OUT_PATH, "w", encoding="utf-8") as f_out:
    for idx, line in enumerate(f_in):
        item = json.loads(line)

        pred = generate_answer(item["prompt_messages"])

        row = {
            "id": idx,
            "prompt_messages": item["prompt_messages"],
            "target_answer": item["target_answer"],
            "model_answer": pred,
            "category": item.get("category", "unknown")
        }

        f_out.write(json.dumps(row, ensure_ascii=False) + "\n")
        print(f"Done {idx + 1}/50")

print(f"Saved outputs to: {OUT_PATH}")

Done 1/50
Done 2/50
Done 3/50
Done 4/50
Done 5/50
Done 6/50
Done 7/50
Done 8/50
Done 9/50
Done 10/50
Done 11/50
Done 12/50
Done 13/50
Done 14/50
Done 15/50
Done 16/50
Done 17/50
Done 18/50
Done 19/50
Done 20/50
Done 21/50
Done 22/50
Done 23/50
Done 24/50
Done 25/50
Done 26/50
Done 27/50
Done 28/50
Done 29/50
Done 30/50
Done 31/50
Done 32/50
Done 33/50
Done 34/50
Done 35/50
Done 36/50
Done 37/50
Done 38/50
Done 39/50
Done 40/50
Done 41/50
Done 42/50
Done 43/50
Done 44/50
Done 45/50
Done 46/50
Done 47/50
Done 48/50
Done 49/50
Done 50/50
Saved outputs to: outputs/eval_50_output.jsonl


In [ ]:
def preview_outputs(path, n=5):
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break

            item = json.loads(line)

            print("=" * 100)
            print("ID:", item["id"])
            print("category:", item["category"])
            for msg in item["prompt_messages"]:
                print(f"{msg['role'].upper()}: {msg['content'][:700]}")

            print("\n target answer:")
            print(item["target_answer"][:1000])

            print("\n model answer:")
            print(item["model_answer"][:1000])

preview_outputs(OUT_PATH, n=5)

ID: 0
category: classification
SYSTEM: You are a helpful assistant.
USER: Classify each of the following as either a city, or a state/province, or neither: San Jose, Shanghai, Jiangsu, Texas, Japan, Shandong

 target answer:
The following are cities: San Jose, Shanghai. The following are States/Provinces: Jiangsu, Texas, Shandong. Japan is a country, so not a city or state/province.

 model answer:
San Jose is a city.
Shanghai is a city.
Jiangsu is a province.
Texas is a state.
Japan is an island country and not a city or a province.
Shandong is a province.
ID: 1
category: summarization
SYSTEM: You are a helpful assistant.
USER: Context:
Lionel Messi is an Argentine professional footballer who has represented the Argentina national football team as a forward since his debut in 2005. Since then, Messi has scored 102 goals in 174 international appearances, making him the country's all-time top scorer; he surpassed Gabriel Batistuta's record of 54 goals with a free kick against the United

#Initialization Model With LORA & Training



In [ ]:
# import gc

# del model
# gc.collect()
# torch.cuda.empty_cache()

In [ ]:
DATA_DIR = Path("data/dolly_sft_full")
TRAIN_PATH = str(DATA_DIR / "train.jsonl")
VALID_PATH = str(DATA_DIR / "valid.jsonl")

OUTPUT_DIR = "outputs/qwen2_5_1_5b_dolly_full_lora_r32"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

tokenizer.pad_token = tokenizer.eos_token


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
model.config.use_cache = False

In [ ]:
train_dataset = load_dataset("json", data_files=TRAIN_PATH, split="train")
eval_dataset = load_dataset("json", data_files=VALID_PATH, split="train")

In [ ]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

In [ ]:
args = SFTConfig(
    output_dir=OUTPUT_DIR,

    max_length=1024,
    packing=False,

    num_train_epochs=1,
    learning_rate=1e-4,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,

    fp16=True,
    bf16=False,

    optim="adamw_torch",
    report_to="none",

    assistant_only_loss=True,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

/tmp/ipykernel_1608/2676387919.py:1: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  args = SFTConfig(


In [ ]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)


Tokenizing train dataset:   0%|          | 0/13000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("saved adapter to:", OUTPUT_DIR)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
200,1.636467,1.534517,1.519603,0.649127,614940.000000
400,1.733455,1.524566,1.535448,0.650213,1208223.000000
600,1.612636,1.520184,1.538254,0.650244,1822342.000000
800,1.751729,1.518353,1.529037,0.651713,2433837.000000
813,1.751729,1.518271,1.528725,0.651859,2471949.000000


saved adapter to: outputs/qwen2_5_1_5b_dolly_full_lora_r32


# Test The Model after Trining

In [ ]:
FT_OUT_PATH = OUT_DIR / "ft_qwen2_5_1_5b_eval_50.jsonl"

In [ ]:
model = trainer.model
model.eval()

def generate_answer_after_ft(prompt_messages):
    text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [ ]:
with open(EVAL_PATH, "r", encoding="utf-8") as f_in, open(FT_OUT_PATH, "w", encoding="utf-8") as f_out:
    for idx, line in enumerate(f_in):
        item = json.loads(line)

        pred = generate_answer_after_ft(item["prompt_messages"])

        row = {
            "id": idx,
            "prompt_messages": item["prompt_messages"],
            "target_answer": item["target_answer"],
            "model_answer": pred,
            "category": item.get("category", "unknown")
        }

        f_out.write(json.dumps(row, ensure_ascii=False) + "\n")
        print(f"Done {idx + 1}/50")

print(f"Saved fine-tuned outputs to: {FT_OUT_PATH}")

Done 1/50
Done 2/50
Done 3/50
Done 4/50
Done 5/50
Done 6/50
Done 7/50
Done 8/50
Done 9/50
Done 10/50
Done 11/50
Done 12/50
Done 13/50
Done 14/50
Done 15/50
Done 16/50
Done 17/50
Done 18/50
Done 19/50
Done 20/50
Done 21/50
Done 22/50
Done 23/50
Done 24/50
Done 25/50
Done 26/50
Done 27/50
Done 28/50
Done 29/50
Done 30/50
Done 31/50
Done 32/50
Done 33/50
Done 34/50
Done 35/50
Done 36/50
Done 37/50
Done 38/50
Done 39/50
Done 40/50
Done 41/50
Done 42/50
Done 43/50
Done 44/50
Done 45/50
Done 46/50
Done 47/50
Done 48/50
Done 49/50
Done 50/50
Saved fine-tuned outputs to: outputs/ft_qwen2_5_1_5b_eval_50.jsonl


In [ ]:
BASE_PATH = OUT_DIR / "eval_50_output.jsonl"

In [ ]:
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

base_rows = read_jsonl(BASE_PATH)
ft_rows = read_jsonl(FT_OUT_PATH)

for i in range(10):
    base = base_rows[i]
    ft = ft_rows[i]

    print("=" * 100)
    print("ID:", i)
    print("category:", ft["category"])
    print()
    for msg in ft["prompt_messages"]:
        print(f"{msg['role'].upper()}: {msg['content']}")

    print("\nTarget:")
    print(ft["target_answer"])

    print("\nBase model answer:")
    print(base["model_answer"])

    print("\nFine Tuned model answer:")
    print(ft["model_answer"])

ID: 0
category: classification

SYSTEM: You are a helpful assistant.
USER: Classify each of the following as either a city, or a state/province, or neither: San Jose, Shanghai, Jiangsu, Texas, Japan, Shandong

Target:
The following are cities: San Jose, Shanghai. The following are States/Provinces: Jiangsu, Texas, Shandong. Japan is a country, so not a city or state/province.

Base model answer:
San Jose is a city.
Shanghai is a city.
Jiangsu is a province.
Texas is a state.
Japan is an island country and not a city or a province.
Shandong is a province.

Fine Tuned model answer:
San Jose is a city in California.
Shanghai is a city in China.
Jiangsu is a province in China.
Texas is a state in the United States.
Japan is a country.
Shandong is a province in China.
ID: 1
category: summarization

SYSTEM: You are a helpful assistant.
USER: Context:
Lionel Messi is an Argentine professional footballer who has represented the Argentina national football team as a forward since his debut in 2